# J2-02 · Human-in-the-loop : `interrupt()` et `Command(resume=…)`

**Formation « Orchestration d'agents & Monitoring : LangGraph + Langfuse (stack 100 % locale) »**
Jour 2 · créneau 10h00–12h00 · ⏱️ ~2h (dont l'exercice 2.1, 45 min)

| | |
|---|---|
| 🎯 **Objectifs** | 1. Mettre un graphe **en pause** pour une décision humaine · 2. **Reprendre** avec la décision · 3. Éviter le piège du **rejeu** (effets de bord en double) · 4. Gérer des décisions riches (approuver, modifier, rejeter) · 5. Utiliser le `HumanInTheLoopMiddleware` d'un `create_agent` |
| 🧱 **Fil rouge** | Avant de créer un ticket **P1** (astreinte, escalade), HelpDesk-Ops demande l'accord d'un responsable |
| ✅ **Prérequis** | J2-01 (checkpointer, `thread_id`) |
| 📦 **Versions** | `langgraph 1.2` · `langchain 1.4` · `langfuse 4.15`, figées par `uv.lock` |

## 1. Mise en route

### [Code] Le socle du fil rouge : `src/helpdesk_ops`

Le code écrit et expliqué en J1-01 (lecture de `.env`, masquage des données personnelles, client Langfuse, création du modèle) est rangé dans le paquet **`helpdesk_ops`**, installé par `uv sync`. Chaque notebook reste **autonome** : la cellule suivante suffit.

> 💡 **Le « Tips » du Formateur :** le paquet est installé en mode *éditable* : une modification dans `src/helpdesk_ops` est prise en compte après *Kernel → Restart*. Le client Langfuse est un singleton par clé publique : en cas de doute, redémarrez le kernel.

In [ ]:
import os

from langfuse import observe, propagate_attributes

from helpdesk_ops.config import creer_llm
from helpdesk_ops.observabilite import init_observabilite
from helpdesk_ops.texte import sans_accents

langfuse, langfuse_handler = init_observabilite()   # .env chargé, masquage PII actif, clés vérifiées
llm = creer_llm("agent")                             # MODEL_AGENT, temperature=0, num_ctx=16384
print(f"✅ Langfuse : {os.environ['LANGFUSE_BASE_URL']} | modèle : {llm.model}")

## 2. Pourquoi une validation humaine ?

### [Théorie] La signature du responsable

*Analogie :* au-delà d'un certain montant, un bon de commande attend la **signature du responsable**. Le processus ne s'arrête pas : il **attend**, puis reprend exactement où il en était.

Un agent qui **agit** (créer un ticket P1 qui réveille l'astreinte, redémarrer un serveur, envoyer un mail) doit pouvoir s'arrêter avant l'action irréversible. C'est la *supervision humaine* : obligatoire pour les systèmes d'IA à haut risque (AI Act, art. 14), et bonne pratique pour tous les autres.

| Sans HITL | Avec HITL |
|---|---|
| l'agent décide et agit seul | l'agent **propose**, un humain **dispose** |
| une erreur de priorité réveille l'astreinte à 3 h | le responsable corrige P1 → P2 avant la création |
| responsabilité floue | décision tracée : qui a validé, quand, quoi |

## 3. La mécanique : `interrupt()` met en pause, `Command(resume=…)` reprend

### [Théorie] Trois règles

1. `interrupt(valeur)` **suspend** le graphe et rend `valeur` à l'appelant (la question posée à l'humain) ;
2. l'état est **sauvegardé** dans le checkpoint : il faut un **checkpointer** et un `thread_id` ;
3. `graphe.invoke(Command(resume=reponse), config)` reprend : dans le nœud, `interrupt()` **renvoie** `reponse`.

On l'illustre sans LLM : la mécanique est la même, et l'exécution instantanée.

In [ ]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class Demande(TypedDict):
    resume: str
    priorite: str
    decision: str


def valider(state: Demande) -> dict:
    reponse = interrupt({"question": f"Créer le ticket {state['priorite']} « {state['resume']} » ?"})
    return {"decision": reponse}


def executer(state: Demande) -> dict:
    print(f"   ⚙️ exécution : décision = {state['decision']}")
    return {}


b = StateGraph(Demande)
b.add_node("valider", valider)
b.add_node("executer", executer)
b.add_edge(START, "valider")
b.add_edge("valider", "executer")
b.add_edge("executer", END)
mini = b.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "j2-02-mini"}}
sortie = mini.invoke({"resume": "ERP de production arrêté", "priorite": "P1"}, config, version="v2")
print("Interruption :", sortie.interrupts[0].value)
print("En attente de :", mini.get_state(config).next)

**👀 Ce que vous devez observer :** l'appel **rend la main** avec une interruption ; `next` vaut `('valider',)` : le graphe attend. Rien n'est perdu : on pourrait reprendre dans une heure, depuis un autre processus (avec `SqliteSaver`).

In [ ]:
sortie = mini.invoke(Command(resume="approuvé par resp-n2"), config, version="v2")
print("État final :", sortie.value, "| interruptions restantes :", sortie.interrupts)

## 4. ⚠️ Le piège du rejeu

### [Théorie] À la reprise, le nœud repart **du début**

LangGraph ne sait pas reprendre au milieu d'une fonction Python : il **réexécute le nœud** depuis sa première ligne, et cette fois `interrupt()` renvoie la réponse au lieu de suspendre. Tout ce qui précède `interrupt()` s'exécute donc **deux fois**.

In [ ]:
compteur = {"tickets_crees": 0}


def valider_et_creer_MAUVAIS(state: Demande) -> dict:
    compteur["tickets_crees"] += 1          # ❌ effet de bord AVANT interrupt : rejoué à la reprise
    reponse = interrupt({"question": "Valider ?"})
    return {"decision": reponse}


b = StateGraph(Demande)
b.add_node("valider", valider_et_creer_MAUVAIS)
b.add_edge(START, "valider")
b.add_edge("valider", END)
mauvais = b.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "j2-02-rejeu"}}
mauvais.invoke({"resume": "ERP", "priorite": "P1"}, config)
mauvais.invoke(Command(resume="ok"), config)
print("Tickets créés pour UNE demande :", compteur["tickets_crees"])

**👀 Ce que vous devez observer :** **2** tickets pour une seule demande.

**Les trois parades, à combiner :**

| Parade | Comment |
|---|---|
| Ordre | aucun effet de bord **avant** `interrupt()` dans un nœud |
| Découpage | l'action dans un **nœud séparé**, après la validation (`valider → creer_ticket`) |
| Idempotence | l'action reçoit une **clé** (ici le `thread_id`) : la rejouer renvoie le résultat existant |

> 💡 **Le « Tips » du Formateur :** la même règle vaut pour le *time travel* (J2-01) et les *retries* (J2-04). Tout nœud peut être rejoué : un outil qui écrit doit être **idempotent**. C'est la règle de production la plus souvent oubliée.

In [ ]:
from helpdesk_ops.ticketing import Ticketing

ticketing = Ticketing()
for _ in range(3):   # trois rejeux de la même demande (même thread)
    ticket = ticketing.creer(cle_idempotence="j2-02-rejeu", categorie="logiciel", priorite="P1",
                             resume="ERP arrêté", file="N1")
print("Tickets en base :", [t.id for t in ticketing.lister()])

## 5. Des décisions riches : approuver, modifier, rejeter

### [Théorie] La réponse humaine est une **donnée**, pas une confiance aveugle

La valeur passée à `Command(resume=…)` vient d'une interface, d'un e-mail, d'un chat : on la **valide** comme n'importe quelle entrée. Une réponse invalide repose la question (une boucle autour d'`interrupt()`).

In [ ]:
from typing import Literal

ACTIONS = ("approuver", "modifier", "rejeter")


class Ticket(TypedDict, total=False):
    resume: str
    priorite: str
    validation: dict


def validation_humaine(state: Ticket) -> Command[Literal["creer", "refuser"]]:
    question = {"ticket_propose": {"resume": state["resume"], "priorite": state["priorite"]}, "actions": ACTIONS}
    while True:   # chaque interrupt() de la boucle attend sa propre reprise
        reponse = interrupt(question)
        if isinstance(reponse, dict) and reponse.get("action") in ACTIONS and reponse.get("par"):
            break
        question = {**question, "erreur": f"réponse invalide : {reponse!r} (attendu : action + par)"}
    if reponse["action"] == "rejeter":
        return Command(update={"validation": reponse}, goto="refuser")
    maj = {"validation": reponse}
    if reponse["action"] == "modifier":
        maj["priorite"] = reponse.get("priorite", state["priorite"])
    return Command(update=maj, goto="creer")


def creer(state: Ticket) -> dict:
    print(f"   ✅ ticket {state['priorite']} créé, validé par {state['validation']['par']}")
    return {}


def refuser(state: Ticket) -> dict:
    print(f"   ⛔ refusé par {state['validation']['par']} : {state['validation'].get('motif', '-')}")
    return {}


b = StateGraph(Ticket)
for nom, fn in (("validation_humaine", validation_humaine), ("creer", creer), ("refuser", refuser)):
    b.add_node(nom, fn)
b.add_edge(START, "validation_humaine")
b.add_edge("creer", END)
b.add_edge("refuser", END)
circuit = b.compile(checkpointer=InMemorySaver())

config = {"configurable": {"thread_id": "j2-02-riche"}}
s = circuit.invoke({"resume": "Wi-Fi coupé au 3e étage", "priorite": "P1"}, config, version="v2")
s = circuit.invoke(Command(resume="oui oui"), config, version="v2")   # réponse invalide
print("Question reposée :", s.interrupts[0].value["erreur"])
s = circuit.invoke(Command(resume={"action": "modifier", "priorite": "P2", "par": "resp-n2"}), config, version="v2")

**👀 Ce que vous devez observer :** la réponse « oui oui » est refusée et la question reposée avec l'erreur ; la décision « modifier » crée un ticket **P2**, validé par `resp-n2`.

## 6. HITL dans un agent LLM : le `HumanInTheLoopMiddleware`

### [Théorie] La même mécanique, empaquetée pour `create_agent`

Pour un `create_agent`, inutile d'écrire le nœud : le middleware intercepte les **appels d'outils** listés et lève une interruption avant leur exécution. Décisions possibles : `approve`, `edit`, `reject`, `respond`. Le principe des middlewares est présenté au red-team du J4 ; les autres middlewares fournis sont hors programme de cette session.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain_core.tools import tool

from helpdesk_ops.outils import statut_serveur

journal_tickets = Ticketing()


@tool(parse_docstring=True)
def creer_ticket(resume: str, priorite: str) -> str:
    """Crée un ticket dans l'outil de ticketing NordTech (action réelle, irréversible).

    Args:
        resume: résumé du problème en une phrase, sans donnée personnelle
        priorite: 'P1' (production arrêtée) à 'P4' (simple demande)
    """
    t = journal_tickets.creer(cle_idempotence=resume, categorie="autre", priorite=priorite, resume=resume, file="N1")
    return f"ticket {t.id} créé"


agent = create_agent(
    model=llm,
    tools=[statut_serveur, creer_ticket],
    system_prompt="Tu es l'agent support N1 de NordTech SA. Vérifie l'état du serveur, puis crée un ticket si nécessaire.",
    checkpointer=InMemorySaver(),
    middleware=[HumanInTheLoopMiddleware(interrupt_on={"creer_ticket": True, "statut_serveur": False})],
)

config = {"configurable": {"thread_id": "j2-02-middleware"}, "callbacks": [langfuse_handler], "recursion_limit": 10}
with propagate_attributes(session_id="j2-02-middleware", tags=["jour-2", "hitl"]):
    s = agent.invoke({"messages": [{"role": "user", "content": "srv-erp-02 est en panne et toute la compta est bloquée, ouvre un ticket P1."}]},
                     config, version="v2")
if s.interrupts:
    for demande in s.interrupts[0].value["action_requests"]:
        print("Validation demandée :", demande["name"], demande["args"])
else:
    print("Pas d'interruption : le modèle n'a pas appelé creer_ticket →", s.value["messages"][-1].content)

In [ ]:
if s.interrupts:
    with propagate_attributes(session_id="j2-02-middleware", tags=["jour-2", "hitl"]):
        s = agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config, version="v2")
    print("Réponse :", s.value["messages"][-1].content)
    print("Tickets :", [(t.id, t.priorite) for t in journal_tickets.lister()])
langfuse.flush()

**👀 Ce que vous devez observer :**
- `statut_serveur` s'exécute librement (lecture seule) ; `creer_ticket` **attend** ;
- dans Langfuse, **Sessions → `j2-02-middleware`** : **deux** traces, l'une s'arrête sur l'interruption, l'autre reprend. La décision humaine n'apparaît pas comme une génération : pour l'audit, journalisez-la (`par`, horodatage) dans l'état ou dans le ticket.

> 🛡️ **Validation humaine ≠ sécurité garantie :** l'humain peut lui-même être manipulé (texte de validation trompeur, fatigue de validation). Montrez-lui les **arguments exacts** de l'action, pas un résumé rédigé par le LLM.

## 7. 🧪 Exercice 2.1 (45 min)

➡️ `exercices/enonces/Ex2_1_validation_p1.ipynb` : un ticket **P1** exige une approbation humaine, les autres sont créés directement, sans doublon.

## 🧭 Récap

| Vous savez maintenant… | API |
|---|---|
| Suspendre un graphe | `interrupt(valeur)` (+ checkpointer + `thread_id`) |
| Lire l'interruption | `invoke(..., version="v2").interrupts`, `get_state(config).next` |
| Reprendre | `invoke(Command(resume=reponse), config)` |
| Éviter les doublons | rien avant `interrupt()`, action dans un nœud séparé, idempotence |
| HITL sur un `create_agent` | `HumanInTheLoopMiddleware(interrupt_on={…})`, `{"decisions": [{"type": "approve"}]}` |

### ✅ Quiz express

1. Pourquoi `interrupt()` exige-t-il un checkpointer ?
2. Que se passe-t-il pour le code placé avant `interrupt()` lors de la reprise ?
3. Citez deux parades contre les actions en double.
4. Pourquoi valider la réponse humaine ?

<details>
<summary>Réponses</summary>

1. Le graphe s'arrête : l'état doit être sauvegardé pour reprendre plus tard, éventuellement dans un autre processus.
2. Il est réexécuté : le nœud repart de sa première ligne.
3. Aucun effet de bord avant `interrupt()` ; action dans un nœud séparé ; clé d'idempotence.
4. C'est une entrée externe comme une autre : format invalide, erreur de saisie, voire manipulation.
</details>